# CV Match Analysis — Colab GPU run

Runs the perception + assisted-tagging pipeline on a Colab **GPU** (escapes the local M1 16 GB memory wall) and renders an **annotated showcase video** of what the pipeline currently does.

**Steps**
1. `Runtime → Change runtime type → GPU` (T4 is enough).
2. Upload this notebook to Colab (`File → Upload notebook`).
3. `Runtime → Run all`. Cell 1 loads the code from a **zip you upload** (`cv_match_analysis.zip`); the weights + sample clip are downloaded in cell 3 (no Roboflow key needed).

No GitHub access required — everything comes from the uploaded zip.

In [ ]:
# 0) Confirm a GPU is attached
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime>GPU')

In [ ]:
# 1) Load the code from the uploaded zip. This cell handles both ways of getting it in:
#    (a) you already unzipped it via the Files panel -> it just finds and cds into it;
#    (b) otherwise it opens an upload dialog -> pick cv_match_analysis.zip, it unzips.
import os, glob, zipfile

def _find_root():
    # a project root is any dir that contains both run.py and src/
    for p in glob.glob('/content/**/run.py', recursive=True):
        d = os.path.dirname(p)
        if os.path.isdir(os.path.join(d, 'src')):
            return d
    return None

root = _find_root()
if root is None:
    from google.colab import files
    print('Pick cv_match_analysis.zip ...')
    up = files.upload()
    for name in up:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall('/content/code')
    root = _find_root()
assert root, 'No run.py + src/ found — upload cv_match_analysis.zip and re-run this cell.'
os.chdir(root)              # absolute chdir: survives re-runs / runtime restarts
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))

In [ ]:
# 2) Install deps. We keep Colab's preinstalled numpy (2.x) to AVOID a runtime restart
#    (our requirements pin numpy<2 for local arm64; on Colab numpy 2 is fine).
!grep -ivE '^numpy' requirements.txt > /tmp/reqs_colab.txt
!pip install -q -r /tmp/reqs_colab.txt
print('deps installed (numpy kept at Colab default)')

In [ ]:
# 3) Download pretrained weights + one DFL sample clip (no Roboflow API key needed)
!python scripts/download_assets.py
!ls -lh data/models data/raw_matches

In [ ]:
# 4) Run the pipeline on GPU. 500 frames (~20s) is enough to surface real events;
#    drop --max-frames for the full clip (fast on a T4). First run downloads SigLIP (~1.5GB).
!PYTORCH_ENABLE_MPS_FALLBACK=1 python run.py \
    --source data/raw_matches/2e57b9_0.mp4 \
    --device cuda --max-frames 500 \
    --output-dir outputs/colab

In [ ]:
# 5) Inspect the run summary + the event log the frontend/DB will consume
import json, pandas as pd
print(json.dumps(json.load(open('outputs/colab/run_summary.json')), indent=2))
events = json.load(open('outputs/colab/events.json'))
df = pd.DataFrame(events)
print(f'\n{len(df)} events')
if len(df):
    print(df['type'].value_counts().to_dict())
    cols = ['type','team_id','start_ts','end_ts','length_m','confidence','review_status']
    display(df[[c for c in cols if c in df.columns]].head(25))

In [ ]:
# 6) Visual sanity check: detections + tracks drawn on a few frames
import cv2, supervision as sv, matplotlib.pyplot as plt
from ultralytics import YOLO
model = YOLO('data/models/players.pt')
cap = cv2.VideoCapture('data/raw_matches/2e57b9_0.mp4')
box, lab = sv.BoxAnnotator(), sv.LabelAnnotator()
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, fno in zip(axes.ravel(), [100, 200, 300, 400]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fno)
    ok, img = cap.read()
    if not ok:
        continue
    det = sv.Detections.from_ultralytics(model.predict(img, conf=0.3, device=0, verbose=False)[0])
    ann = lab.annotate(box.annotate(img.copy(), det), det)
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB)); ax.set_title(f'frame {fno}'); ax.axis('off')
cap.release(); plt.tight_layout(); plt.show()

## Annotated showcase + tactical videos

One perception pass over **every frame** (smooth playback at the clip's native fps) produces six videos in `outputs/showcase/`:

**On the broadcast video**
- `showcase.mp4` — team-coloured tracks + ids (GK/ref flagged), ball, HUD (clock, segment, calibration bar, ball status) and the live **event log** (confidence + `accept`/`review`/`hidden`). *(pitch-keypoint overlay removed.)*
- `triangulation_overlay.mp4` — the **Delaunay team-shape mesh** on real player positions.
- `voronoi_overlay.mp4` — **Voronoi control regions** warped onto the actual field via the calibration homography (gated on calibration confidence).

**Top-down 2D maps**
- `radar_voronoi.mp4`, `radar_triangulation.mp4`, `radar_combined.mp4`.

Cell 8 renders them (perception is cached → re-render any subset instantly with `--from-cache --only ...`), cell 9 previews a few inline, cell 10 downloads them all as one zip.

In [ ]:
# 8) Render the SHOWCASE + tactical videos on GPU. Renders EVERY frame, so playback is
#    smooth at the clip's native fps (25 -> 25 fps out). No --max-frames = the whole clip.
#    NOTE on smoothness: output fps = source fps / sampling stride. Sampling coarser
#    (e.g. --interval-s 0.1 -> 12.5 fps) is choppier, so we sample every frame here.
#    Produces 6 mp4s in outputs/showcase/. Perception is cached to frame_states.pkl ->
#    re-render any subset with, e.g.:  --from-cache --only voronoi,radar_combined
!python scripts/render_video.py \
    --source data/raw_matches/2e57b9_0.mp4 \
    --device cuda \
    --output-dir outputs/showcase
!ls -lh outputs/showcase/*.mp4

In [ ]:
# 9) Preview a few inline. Transcode to browser-friendly H.264 + downscale (previews are
#    written outside outputs/showcase so they're not picked up by the download zip).
import subprocess, base64, os
from IPython.display import HTML, display

def show(name, w=900):
    path = os.path.join('outputs/showcase', name)
    if not os.path.exists(path):
        print('missing', path); return
    prev = '/content/_prev_' + name
    subprocess.run(['ffmpeg', '-y', '-i', path, '-vf', 'scale=900:-2', '-vcodec', 'libx264',
                    '-pix_fmt', 'yuv420p', '-movflags', '+faststart', prev],
                   check=True, capture_output=True)
    b64 = base64.b64encode(open(prev, 'rb').read()).decode()
    display(HTML(f'<h4>{name}</h4><video width={w} controls autoplay loop muted>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

for name in ['showcase.mp4', 'voronoi_overlay.mp4', 'triangulation_overlay.mp4', 'radar_combined.mp4']:
    show(name)

In [ ]:
# 10) Download all six videos as a single zip for the pre-MVP showcase tool
import zipfile, glob, os
zpath = '/content/cv_match_analysis_videos.zip'
mp4s = sorted(glob.glob('outputs/showcase/*.mp4'))
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in mp4s:
        z.write(f, arcname=os.path.basename(f))
print('zipped:', [os.path.basename(f) for f in mp4s])
from google.colab import files
files.download(zpath)